<a href="https://www.kaggle.com/code/fredericnicholson/mushroom-soup-with-polars-one-hot-encoding-and-nn?scriptVersionId=238811157" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import polars as pl 
pl.Config (tbl_rows=30)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



In [ ]:
train_df = pl.scan_csv('/kaggle/input/playground-series-s4e8/train.csv')
test_df = pl.scan_csv('/kaggle/input/playground-series-s4e8/test.csv')
train_df.describe ()

# EDA 

the biggest problem is the large number of missing or corrupted feature data. Using the original data set description we  filter out all data out of this range and give it a special class "NN" 

I  will use bining for all numerical data, as so that the NN  is receiving one hot encoding for all input. missing data is a class of its own.  



# Data cleaning and transformation 

In [ ]:
raw_encodings = [
"cap-shape (n): bell=b, conical=c, convex=x, flat=f,sunken=s, spherical=p, others=o",
"cap-surface (n): fibrous=i, grooves=g, scaly=y, smooth=s,shiny=h, leathery=l, silky=k, sticky=t,wrinkled=w, fleshy=e",
"cap-color (n): brown=n, buff=b, gray=g, green=r, pink=p,purple=u, red=e, white=w, yellow=y, blue=l,orange=o, black=k",
"does-bruise-or-bleed (n): bruises-or-bleeding=t,no=f",
"gill-attachment (n): adnate=a, adnexed=x, decurrent=d, free=e,sinuate=s, pores=p, none=f, unknown=?",
"gill-spacing (n): close=c, distant=d, none=f",
"gill-color (n):  brown=n, buff=b, gray=g, green=r, pink=p,purple=u, red=e, white=w, yellow=y, blue=l,orange=o, black=k, none=f",
"stem-root (n): bulbous=b, swollen=s, club=c, cup=u, equal=e,rhizomorphs=z, rooted=r",
"stem-surface (n):  brown=n, buff=b, gray=g, green=r, pink=p,purple=u, red=e, white=w, yellow=y, blue=l,orange=o, black=k,none=f",
"stem-color (n):  brown=n, buff=b, gray=g, green=r, pink=p,purple=u, red=e, white=w, yellow=y, blue=l,orange=o, black=k, none=f",
"veil-type (n): partial=p, universal=u",
"veil-color (n):  brown=n, buff=b, gray=g, green=r, pink=p,purple=u, red=e, white=w, yellow=y, blue=l,orange=o, black=k, none=f",
"has-ring (n): ring=t, none=f",
"ring-type (n): cobwebby=c, evanescent=e, flaring=r, grooved=g,large=l, pendant=p, sheathing=s, zone=z, scaly=y, movable=m, none=f, unknown=?",
"spore-print-color (n):  brown=n, buff=b, gray=g, green=r, pink=p,purple=u, red=e, white=w, yellow=y, blue=l,orange=o, black=k",
"habitat (n): grasses=g, leaves=l, meadows=m, paths=p, heaths=h,urban=u, waste=w, woods=d",
"season (n): spring=s, summer=u, autumn=a, winter=w"
]



In [ ]:
class Cleaning :
    def parse_encodings (self, description) :
       column_name = description.split (' ', 1)[0]
       left_over = description.split (' ', 1)[1]
       encoding_pairs = left_over.split (',')
       encodings  = [value.split ("=")[1] for value in encoding_pairs]
       return column_name, encodings 

    def __init__(self, raw_encodings) :
        self.encoding_dict =  {}
        for raw in raw_encodings : 
           col, values = self.parse_encodings (raw)
           self.encoding_dict [col] = values
    
    def transform (self, df : pl.LazyFrame) -> pl.LazyFrame : 
        result = df
        for col in self.encoding_dict.keys() :
            result = result.with_columns (pl.when (pl.col (col).is_in (self.encoding_dict [col])).then (
                pl.col(col)). otherwise (pl.lit('NN')).alias (col))
        return result 
 

In [ ]:
   
cleaner = Cleaning (raw_encodings)    
   
train_df = cleaner.transform (train_df)
                                  
train_df.collect()                                  
                                  

In [ ]:
all_features = train_df.collect_schema().names()
all_features.remove('id')
all_features.remove('class')
all_features

In [ ]:
num_features = ['cap-diameter', 'stem-height', 'stem-width' ]
cat_features = [ f for f in all_features if not f in num_features ]


In [ ]:
#%%time

#from sklearn import preprocessing

#all_feature_data = pl.concat ([train_df.select (all_features), test_df.select(all_features)])

#scaler = preprocessing.StandardScaler().fit(all_feature_data.select (num_features).collect())

In [ ]:
import seaborn as sns

sns.histplot (train_df.collect(), x='cap-diameter', binwidth = 2, hue = 'class') 

In [ ]:
sns.histplot (train_df.collect(), x='stem-height', binwidth = 2, hue = 'class') 

In [ ]:
sns.histplot (train_df.collect(), x='stem-width', binwidth = 5, hue = 'class') 

In [ ]:
sns.countplot (data = train_df.collect().to_pandas(), x = 'stem-color', hue = 'class')

In [ ]:
sns.countplot (data = train_df.collect().to_pandas(), x = 'season', hue = 'class')

In [ ]:
def conv_num_to_one_hot (lazy_df : pl.LazyFrame, column : str, boundaries : list, with_Null  : bool = False) -> pl.LazyFrame :
    print (f"NaN for column {column} before: {lazy_df.select (column).null_count().collect().item(0,0)}")    
    result = lazy_df
    # dealing with Nan and Null 
    if with_Null :
        result = result.with_columns ((pl.col(column).is_null()).or_ (
                                       pl.col(column).is_nan()).cast (pl.Int8).alias (f"{column}_Null"))
        result = result.with_columns(pl.col(column).fill_nan (boundaries [0]))                              
        result = result.with_columns(pl.col(column).fill_null (boundaries [0]))                              
    
    print (f"NaN for column {column} after : {result.select (column).null_count().collect().item(0,0)}")    
    result = result.with_columns ((pl.col(column) <= boundaries [0]).cast (pl.Int8).alias (f"{column}_below") )
    if with_Null :
        result = result.with_columns (pl.when (pl.col(f"{column}_Null").cast(bool)).then (
                     pl.lit(0)).otherwise (
                     pl.col (f"{column}_below") ))
    
    for lower_boundary, upper_boundary in zip (boundaries, boundaries [1:]) :
        result = result.with_columns (pl.col(column).is_between (lower_boundary, upper_boundary).cast (pl.Int8).alias (f"{column}_{lower_boundary}") )
    result = result.with_columns ((pl.col(column) > boundaries [-1]).cast (pl.Int8).alias (f"{column}_above") )
    result = result.drop (column)
    return result



In [ ]:
class Lazy_one_hot_encoding :
    def __init__(self, limit = 500) :
        self.new_features = []
        self.old_features = []
        self.original_features = []
        self.limit = limit
    
    def fit  (self, df_lazy : pl.LazyFrame, cols : list) -> pl.LazyFrame:
        # limit eliminates categories that are not used very often  
        self.old_features = df_lazy.collect_schema().names()
        self.original_features = cols
        # lazy frame does not support one hot encoding, so we need to collect in a local variable.
        for c in cols : 
            how_many = df_lazy.group_by(c).len().collect()
            drop_value = False 
            for row   in how_many.iter_rows() :
               value = row [0]
               count = row[1]
               if count > self.limit : 
                   new_column = f'{c}###{value}'   
                   self.new_features.append(new_column)  
               else :
                   drop_value = True
            # this implements teh drop_first option 
            if not drop_value :
                del self.new_features [-1] 
        
        
    def transform  (self, df_lazy) ->  pl.LazyFrame:
        
        result = df_lazy
        for new_feature in self.new_features :
            split = new_feature.split ('###')
            feature = split [0]
            value = split [1]
            try :
                value = float (value)
            except :
                value = value 
                # nothing to do, value is already in the format in the column 
                
            result = result.with_columns ((pl.col(feature) == value).cast(pl.UInt8).alias (new_feature))
        
        return result.drop (self.original_features)


In [ ]:
%%time 

all_cat_data = pl.concat ([cleaner.transform (train_df), cleaner.transform (test_df)] )

one_hot_encoder = Lazy_one_hot_encoding()

one_hot_encoder.fit (all_cat_data, cat_features)

# Defining the pipeline

In [ ]:
%%time
y_train = train_df.collect().get_column ('class').replace_strict ({'e' : 0, 
                                                         'p' : 1})
X_train = train_df.pipe(
                cleaner.transform).pipe(
                conv_num_to_one_hot ,'cap-diameter', [2,4,6,8,10,12,14,16, 18, 20], True).pipe(
                conv_num_to_one_hot ,'stem-height', [2,4,6,8,10,12,14,16, 18, 20], True).pipe(
                conv_num_to_one_hot ,'stem-width', [5,10,15,20,25,30,35,40],True).pipe(
                one_hot_encoder.transform).pipe(
                lambda df : df.drop (['id', 'class'])  )  

In [ ]:
print (f"NaN values in y_train :{ sum (y_train.is_nan())}")
print (f"Null values in X_train :{ X_train.null_count().collect()}")



X_train_np = X_train.collect().to_numpy()
y_train_np = y_train.to_numpy()
                                      

# a stateful MCC calculation for KERAS with batches

In [ ]:
import keras
import tensorflow as tf
@keras.saving.register_keras_serializable()
class MCC(keras.metrics.Metric):
    def __init__(self, name='mcc', **kwargs):
        super().__init__(name=name, **kwargs)
        # create the mcc variable
        self.mcc_ = self.add_variable(
            shape=(),
            initializer='zeros',
            name='mcc'
        )
        # create metric objects and preserve their state to get tp, tn, fp, fn for each batch 
        self.tp = keras.metrics.TruePositives()
        self.tn = keras.metrics.TrueNegatives()
        self.fp = keras.metrics.FalsePositives()
        self.fn = keras.metrics.FalseNegatives()

    def reset_state (self) :
        self.tp.reset_state()
        self.tn.reset_state()
        self.fp.reset_state()
        self.fn.reset_state()
        self.mcc_.assign(tf.zeros(shape = ()))
# this function is crucial as it needs to collect all the       
    def update_state(self, y_true, y_pred, sample_weight=None):
        # as we use several batches, and this function is call after every batch, we need to accumulate the score. 
         
        #compute true positive:
        self.tp.update_state(y_true, y_pred)
        current_tp = self.tp.result()
         
        # true negatives
        self.tn.update_state(y_true, y_pred)
        current_tn = self.tn.result()

        # false positives
        self.fp.update_state(y_true, y_pred)
        current_fp = self.fp.result()

        # false negatives
        self.fn.update_state(y_true, y_pred)
        current_fn = self.fn.result()

        mcc_result = (current_tp * current_tn - current_fp * current_fn) / keras.ops.sqrt(
                       (current_tp + current_fp) * (current_tp + current_fn) * (
                        current_tn + current_fp) * (current_tn + current_fn))

        #Assign the mcc_result to the self.mcc_ variable
        self.mcc_.assign(mcc_result)


    def result(self):
        return self.mcc_

In [ ]:
# testing the mcc function
mcc = MCC()
a = tf.constant ([0,1,1,0,0,1])
b = tf.constant ([0.1,0.7,0.8,0.4,0.8,0.57])
mcc.update_state (a,b)
print (mcc.result().numpy())

mcc.reset_state()

# Neural network training 

In [ ]:
import keras
import tensorflow as tf 
from keras.layers import Dense, Normalization, Dropout
from keras.models import  Sequential
from keras.callbacks import EarlyStopping
from keras.layers import Dense, Dropout, Normalization
import time 
import math 

# prepare fo hyperparameter tuning 

In [ ]:
# def build_model_w_param(num_layer : int, 
#                 neuron_start_layer : int,
#                 neuron_shrink : float) -> keras.Sequential:
#     model = keras.Sequential()
#     model.add (keras.Input(shape = (X_train_np.shape[1],)))
#     num_neuron_in_layer = neuron_start_layer
#     # Tune the number of layers.
#     for i in range(num_layer):
#         print (f'adding layer with {num_neuron_in_layer} neurons')
#         if num_neuron_in_layer > 0 :
#             model.add( Dense(
#                       units = num_neuron_in_layer,  
#                       activation= "relu"))
#         num_neuron_in_layer = math.floor (num_neuron_in_layer * neuron_shrink)   
#         model.add (Normalization(axis=-1))
#     model.add (Dense(units=1, activation = tf.nn.sigmoid))            
#     return model
      
    
    

In [ ]:
# m =  build_model_w_param (3, 14, 0.5)
# m.summary

In [ ]:
# from sklearn.model_selection import StratifiedShuffleSplit
# from sklearn.model_selection import ShuffleSplit
# from sklearn.metrics import matthews_corrcoef

# SPLITS = 2
# EPOCHS = 7
# # PATIENCE = 10

# def    evaluate_network(num_layer : float, 
#                 neuron_start_layer : float,
#                 neuron_shrink : float, 
#                 flex_learning_rate : float, 
#                 batch_size : int) -> float:
#     # Bootstrap
#     num_layer_int = int (num_layer)
#     neuron_start_layer_int = int (neuron_start_layer)
#     batch_size_int = int (batch_size)
    
#     boot = StratifiedShuffleSplit(n_splits=SPLITS, test_size=0.1)
#     # for Regression
#     # boot = ShuffleSplit(n_splits=SPLITS, test_size=0.1)

#     # Track progress
#     benchmark = []
#     # epochs_needed = []
#     num = 0
    
#     # Loop through samples
#     for train, test in boot.split(X_train_np,y_train_np):
#         start_time = time.time()
#         num+=1

#         # Split train and test
#         x_train = X_train_np [train]
#         y_train = y_train_np [train]
#         x_test = X_train_np  [test]
#         y_test = y_train_np [test]

#         model =  build_model_w_param(num_layer_int, neuron_start_layer_int, neuron_shrink)
#         little_adam = tf.keras.optimizers.Adam (learning_rate=flex_learning_rate)

#         model.compile(optimizer= little_adam,
#               loss=tf.keras.losses.BinaryCrossentropy(),
             
#               metrics=[keras.metrics.BinaryAccuracy(),
#                        #cheap_metric,
#                        mcc,
#                       # keras.metrics.F1Score (),
#                       #keras.metrics.Recall (name = 'recall'), 
#                       keras.metrics.FalsePositives (name ='FP'),
#                       keras.metrics.FalseNegatives (name = 'FN')
#                       ])

#         #monitor = EarlyStopping(monitor='val_loss', min_delta=1e-3, 
#         #patience=PATIENCE, verbose=0, mode='auto', 
#         #                        restore_best_weights=True)

#         # Train on the bootstrap sample
#         model.fit(x_train,y_train,validation_data=(x_test,y_test),
#                   # callbacks=[monitor],
#                   verbose=0,epochs=EPOCHS, 
#                   batch_size = batch_size_int )
#         #epochs = monitor.stopped_epoch
#         #epochs_needed.append(epochs)

#         # Predict on the out of boot (validation)
#         y_predict =  np.rint (model.predict(x_test, batch_size = batch_size_int))
        
#         # Measure this bootstrap's log loss
#         mcc_current = matthews_corrcoef (y_test, y_predict)
#         benchmark.append(mcc_current)
#         #m1 = statistics.mean(mean_benchmark)
#         #m2 = statistics.mean(epochs_needed)
#         #mdev = statistics.pstdev(mean_benchmark)

#         # Record this iteration
#         time_took = time.time() - start_time
        
#     tf.keras.backend.clear_session()
    
#     return (np.mean (benchmark)  ) 


In [ ]:
# a = evaluate_network(3,120 , 0.5, 0.001, batch_size = 1024)

# a 

In [ ]:
# !pip install bayesian-optimization

In [ ]:
# from bayes_opt import BayesianOptimization
# import time

# # Supress NaN warnings
# import warnings
# warnings.filterwarnings("ignore",category =RuntimeWarning)

# # Bounded region of parameter space
# pbounds = {'num_layer': (1, 7),
#            'neuron_start_layer': (12, 512),
#            'neuron_shrink': (0.01, 0.8),
#            'flex_learning_rate': (0.000001, 0.01), 
#            'batch_size'  :(32, 1024)
#           }

# optimizer = BayesianOptimization(
#     f=evaluate_network,
#     pbounds=pbounds,
#     verbose=2,  # verbose = 1 prints only when a maximum 
#     # is observed, verbose = 0 is silent
#     random_state=1,
# )

# start_time = time.time()
# optimizer.maximize(init_points=20, n_iter=80,)
# time_took = time.time() - start_time

# print(f"Total runtime: {hms_string(time_took)}")
# print(optimizer.max)

In [ ]:


def build_model () :
    model = keras.Sequential()
    model.add (keras.Input(shape = (X_train_np.shape[1],)))
    model.add (Normalization(axis=-1))
    #model.add (keras.layers.Dropout (0.1) )
    model.add (Dense(units=64, activation="relu"))
    model.add (Normalization(axis=-1))
   #  model.add (keras.layers.Dropout (0.1) )
    model.add (Dense(units=32, activation="relu"))
    model.add (Normalization(axis=-1))
    # model.add (keras.layers.Dropout (0.1) )
    model.add (Dense(units=16, activation="relu"))
    model.add (Normalization(axis=-1))
    # model.add (keras.layers.Dropout (0.1) )
    model.add (Dense(units=8, activation="relu"))
    model.add (Normalization(axis=-1))
    model.add (Dense(units=4, activation="relu"))
    model.add (Normalization(axis=-1))
    model.add (Dense(units=1, activation = tf.nn.sigmoid))
    # model.add (Dense(units=1, activation = tf.nn.softmax))
    # save the initial weights for later
    #initial_weights = model.get_weights()

    # label_smoothing = 0.1 -> 0.8291

    little_adam = tf.keras.optimizers.Adam (learning_rate=0.003687)

    model.compile(optimizer= little_adam,
              loss=tf.keras.losses.BinaryCrossentropy(),
             
              metrics=[keras.metrics.BinaryAccuracy(),
                       #cheap_metric,
                       mcc,
                      # keras.metrics.F1Score (),
                      #keras.metrics.Recall (name = 'recall'), 
                      keras.metrics.FalsePositives (name ='FP'),
                      keras.metrics.FalseNegatives (name = 'FN')
                      ])

    model.build()
    return model 

In [ ]:
from matplotlib import pyplot as plt 

def plot_fold (history) :
    #plt.plot(history.history['val_mcc'])
    plt.plot(history.history['val_FN'])
    plt.plot(history.history['val_FP'])
    plt.title('False predicted')
    plt.ylabel('absolute value')
    plt.xlabel('epoch')
    plt.legend(['False positive', 'false negative'], loc='upper left')
    plt.show()


In [ ]:
stop_on_val_binary_accuracy = keras.callbacks.EarlyStopping(monitor='val_binary_accuracy', 
                                          patience=8, mode = 'max', restore_best_weights = True)
stop_on_val_mcc = keras.callbacks.EarlyStopping(monitor="val_mcc", 
                                                patience=3, 
                                                mode='max',  
                                                verbose = True, min_delta = 0.0001, 
                                                restore_best_weights = True)
stop_on_val_cheap = keras.callbacks.EarlyStopping(monitor='val_cheap_metric', 
                                          patience=8, mode = 'min', restore_best_weights = True)

In [ ]:
# from sklearn.model_selection import train_test_split
# X_fit, X_validation,  y_fit, y_validation = train_test_split(
#     X_train_np, y_train_np, train_size = 0.8, shuffle=True)


# tuner.search(X_fit, y_fit, epochs=5, batch_size = 1024, validation_data=(X_validation, y_validation))

In [ ]:
from sklearn.metrics import matthews_corrcoef

models = []
history_folds = []
mcc_folds = []
from sklearn.model_selection import StratifiedKFold

epoch_switch = 50
skf = StratifiedKFold(n_splits=5)

for i, (train_index, validation_index) in enumerate(skf.split(X_train_np, y_train_np)):
    
    X_train_fold = X_train_np[train_index]
    y_train_fold = y_train_np[train_index]
    X_validation_fold = X_train_np [validation_index]
    y_validation_fold = y_train_np [validation_index]
    print (f'train size : {y_train_fold.shape} for fold {i}')
    model = build_model ()
    
    history = model.fit(x=X_train_fold, y= y_train_fold, epochs=epoch_switch, batch_size = 409, 
                    verbose=0, 
                    #callbacks = [ #stop_on_val_binary_accuracy,
                    #             stop_on_val_mcc 
                                 #stop_on_val_cheap,
                                 #, callback2, callback3, 
                                 # callback4, callback5, callback6,  callback7, callback8,callback9
                    #            ],     
                    validation_data =  (X_validation_fold, y_validation_fold)
                    )
    y_predict = np.rint (model.predict (X_validation_fold, batch_size = 124))
    mcc_current = matthews_corrcoef (y_validation_fold, y_predict)
    mcc_folds.append (mcc_current)
    print (f'MCC (sklearn.metrics) for current fold : {mcc_current}')
    mcc.reset_state()
    mcc.update_state(y_validation_fold, y_predict)
    my_mcc = mcc.result().numpy()
    print (f'class based MCC for current fold : {my_mcc}')
    models.append(model)
    history_folds.append(history)
    
    

In [ ]:
overall_mcc = pl.Series (mcc_folds)

print (f'overall_mcc mean: {overall_mcc.mean()}')
print (f'overall_mcc max: {overall_mcc.max()}')
print (f'overall_mcc min: {overall_mcc.min()}')

best_epoch = overall_mcc.arg_max()
print (f'best_epoch  {best_epoch}')


In [ ]:
 from matplotlib import pyplot as plt 

 for index, history in enumerate (history_folds) :
     #print ( history.history.keys())
    val_mcc_list =  history.history['val_mcc']
    
    for epoch,n in enumerate (val_mcc_list) :
        print (f"epoch {epoch} mcc = {n.numpy()}")  
#     ep  = [ n for n in range (15)]
#     plt.plot([res,ep] )
# plt.title('model mcc')
# plt.ylabel('MCC')
# plt.xlabel('epoch')
# plt.legend(['epoch 0', 'epoch 1', 'epoch 2 ', 'epoch 3', 'epoch 4', 'epoch 5', 'epoch 6 ', 'epoch 7', 'epoch 8', 'epoch 9'], loc='upper left')  
# plt.show()




# submission 

In [ ]:

X_submit  = test_df.pipe(
                cleaner.transform).pipe(
                conv_num_to_one_hot ,'cap-diameter', [2,4,6,8,10,12,14,16, 18, 20], True).pipe(
                conv_num_to_one_hot ,'stem-height', [2,4,6,8,10,12,14,16, 18, 20], True).pipe(
                conv_num_to_one_hot ,'stem-width', [5,10,15,20,25,30,35,40],True).pipe(
                one_hot_encoder.transform).pipe(
                lambda df : df.drop (['id'])  ).collect()  

X_submit

X_submit_np = X_submit.to_numpy()


In [ ]:
predictions = np.zeros ((2_077_964, 1))
print (f' summary shape : {predictions.shape}')   

# we can either average or take the best epoch. the best epoch uses a 80% train 20% validation split

model = models [best_epoch] 
predictions = model.predict (X_submit, batch_size = 1024)


#for model in models :       
#     print (type (model))
#     predict = model.predict (X_submit_np, batch_size = 1024)
     
#     predictions += predict    
    

In [ ]:
predict_df = pl.DataFrame (predictions, schema  = ['predict'])

In [ ]:
predict_df.head(20)

In [ ]:
#predict_df = predict_df.with_columns (pl.when (pl.col('predict') > skf.get_n_splits()/2).then (pl.lit ('p')).otherwise(pl.lit('e')).alias ('class'))
predict_df = predict_df.with_columns (pl.when (pl.col('predict') > 0.5).then (pl.lit ('p')).otherwise(pl.lit('e')).alias ('class'))

print (predict_df.head(20))
submission = pl.concat ([test_df.select ('id').collect(), predict_df], how = 'horizontal')

submission = submission.drop('predict') 

print (submission.head(20))


In [ ]:
submission.write_csv ('submission.csv')

In [ ]:
submission.group_by('class').len()